# ScoreCliff — experiment runner

Runs the full v2 study on a single Kaggle GPU (P100 / T4):

1. **Zero-shot CLIP** baseline on COCO val2017 (5K and 1K galleries)
2. Fine-tuning on 1% of COCO train2017 with **random**, **QuRe** and **cluster** negatives, for several seeds
3. Mean ± std tables, paired bootstrap significance tests, mining diagnostics and qualitative retrievals

**Setup on Kaggle:** attach the dataset `awsaf49/coco-2017-dataset`, turn on a GPU and internet access.
Because the repository is private, either add a Kaggle secret `GITHUB_TOKEN` (a read-only fine-grained token
for this repo) or upload the repo as a Kaggle dataset and set `LOCAL_SRC` below.

Approximate time per training run on a P100: ~10–15 min (5 epochs, 5.9K captions).

In [ ]:
import os, sys, glob, json, subprocess

REPO_URL  = "https://github.com/Lazypilottt/scorecliff.git"
LOCAL_SRC = None                       # e.g. "/kaggle/input/scorecliff-src" if uploaded as a dataset
COCO_ROOT = "/kaggle/input/coco-2017-dataset/coco2017"

SEEDS = [0, 1, 2]
EXPERIMENTS = {                         # name -> extra CLI arguments
    "random":         ["--miner", "random"],
    "qure":           ["--miner", "qure"],
    "cluster-kmeans": ["--miner", "cluster", "--cluster-algo", "kmeans"],
}
COMMON = [
    "--csv", "data/train2017.csv", "--caption-frac", "0.01", "--subset-seed", "501",
    "--epochs", "5", "--batch-size", "64", "--lr", "1e-5",
    "--eval", "5k=data/val2017.csv", "1k=data/val2017.csv:1000",
]
KEEP_CHECKPOINTS = False                # each checkpoint is ~600 MB

In [ ]:
def run(args):
    """Run a command and stream its output into the notebook."""
    args = [str(a) for a in args]
    print("$ " + " ".join(args), flush=True)
    env = {**os.environ, "TQDM_DISABLE": "1"}
    p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    for line in p.stdout:
        print(line, end="")
    if p.wait():
        raise RuntimeError(f"command failed with exit code {p.returncode}")

SC = [sys.executable, "-m", "scorecliff"]

## 1. Install

In [ ]:
if LOCAL_SRC:
    SRC = LOCAL_SRC
else:
    url = REPO_URL
    try:
        from kaggle_secrets import UserSecretsClient
        url = REPO_URL.replace("https://", f"https://{UserSecretsClient().get_secret('GITHUB_TOKEN')}@")
    except Exception:
        pass
    if not os.path.exists("scorecliff-src"):
        subprocess.run(["git", "clone", "--depth", "1", url, "scorecliff-src"], check=True)
    SRC = "scorecliff-src"

run([sys.executable, "-m", "pip", "install", "-q", "open_clip_torch", "scikit-learn>=1.3"])
run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", SRC])
import torch; print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

## 2. Data

In [ ]:
if not os.path.exists(COCO_ROOT):            # outside Kaggle: download (~25 GB)
    import kagglehub
    COCO_ROOT = os.path.join(kagglehub.dataset_download("awsaf49/coco-2017-dataset"), "coco2017")
if not os.path.exists("data/val2017.csv"):
    run(SC + ["prepare", "--coco-root", COCO_ROOT, "--out", "data"])

## 3. Zero-shot baseline

In [ ]:
for name, extra in {"5k": [], "1k": ["--num-images", "1000"]}.items():
    out = f"runs/zero-shot/eval_{name}"
    if not os.path.exists(f"{out}/metrics.json"):
        run(SC + ["evaluate", "--csv", "data/val2017.csv", *extra, "--out", out])

## 4. Fine-tuning runs
Runs are skipped if their results already exist, so this cell can be re-run after a session restart.

In [ ]:
for seed in SEEDS:
    for name, extra in EXPERIMENTS.items():
        out = f"runs/{name}_s{seed}"
        if os.path.exists(f"{out}/eval_1k/metrics.json"):
            continue
        run(SC + ["train", *COMMON, *extra, "--seed", seed, "--out", out])
        if not KEEP_CHECKPOINTS and os.path.exists(f"{out}/model.pt"):
            os.remove(f"{out}/model.pt")

### Optional: all clustering algorithms (one seed)

In [ ]:
RUN_ALL_CLUSTERING = False
if RUN_ALL_CLUSTERING:
    for algo in ["minibatch", "agglomerative", "spectral", "hdbscan"]:
        out = f"runs/cluster-{algo}_s0"
        if not os.path.exists(f"{out}/eval_1k/metrics.json"):
            run(SC + ["train", *COMMON, "--miner", "cluster", "--cluster-algo", algo, "--seed", 0, "--out", out])
            if not KEEP_CHECKPOINTS and os.path.exists(f"{out}/model.pt"):
                os.remove(f"{out}/model.pt")

## 5. Results (mean ± std over seeds)

In [ ]:
for split in ["5k", "1k"]:
    for direction in ["text_to_image", "image_to_text"]:
        print(f"\n### COCO val2017 {split.upper()} — {direction.replace('_', ' ')}\n")
        run(SC + ["summarize", f"runs/*/eval_{split}/metrics.json", "--direction", direction, "--markdown"])

## 6. Significance: paired bootstrap per seed (text → image, 5K)

In [ ]:
pairs = [("qure", "random"), ("cluster-kmeans", "random"), ("qure", "cluster-kmeans")]
for a, b in pairs:
    for seed in SEEDS:
        fa, fb = f"runs/{a}_s{seed}/eval_5k/ranks.npz", f"runs/{b}_s{seed}/eval_5k/ranks.npz"
        if os.path.exists(fa) and os.path.exists(fb):
            run(SC + ["compare", fa, fb, "--out", f"runs/compare_{a}_vs_{b}_s{seed}.json"])
    print()
for name in EXPERIMENTS:
    f = f"runs/{name}_s0/eval_5k/ranks.npz"
    if os.path.exists(f):
        run(SC + ["compare", f, "runs/zero-shot/eval_5k/ranks.npz"])

## 7. Training and mining diagnostics

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4))
for f in sorted(glob.glob("runs/*_s0/history.json")):
    h = json.load(open(f))
    ax.plot([e["epoch"] for e in h["epochs"]], [e["loss"] for e in h["epochs"]], marker="o", label=h["tag"])
ax.set(xlabel="epoch", ylabel="training loss", title="Training loss (seed 0)")
ax.grid(alpha=.3); ax.legend(); plt.show()

h = json.load(open("runs/qure_s0/history.json"))
diag = pd.DataFrame([{"epoch": e["epoch"], **e["mining"]} for e in h["epochs"] if e["mining"].get("miner") == "qure"])
print("QuRe mining statistics per epoch (ranks are 1-based):")
display(diag)

## 8. Qualitative retrievals (5K gallery)

In [ ]:
import numpy as np
from PIL import Image
from scorecliff.data import PairTable

table = PairTable.from_df(pd.read_csv("data/val2017.csv"))
runs = {"zero-shot": "runs/zero-shot/eval_5k", "random": "runs/random_s0/eval_5k", "qure": "runs/qure_s0/eval_5k"}
ranks = {k: np.load(f"{v}/ranks.npz") for k, v in runs.items() if os.path.exists(f"{v}/ranks.npz")}

# queries QuRe gets right at rank 1 that random fine-tuning misses
both = ranks.get("qure"), ranks.get("random")
if all(r is not None for r in both):
    candidates = np.where((both[0]["t2i"] == 1) & (both[1]["t2i"] > 1))[0]
else:
    candidates = np.arange(table.num_captions)
rng = np.random.default_rng(0)
for q in rng.choice(candidates, size=min(4, len(candidates)), replace=False):
    fig, axes = plt.subplots(len(ranks), 6, figsize=(15, 2.6 * len(ranks)))
    axes = np.atleast_2d(axes)
    gt = table.caption_image[q]
    fig.suptitle(f"\"{table.texts[q]}\"", fontsize=11)
    for row, (name, r) in enumerate(ranks.items()):
        axes[row, 0].imshow(Image.open(table.image_paths[gt]).convert("RGB"))
        axes[row, 0].set_title(f"{name}: GT rank {r['t2i'][q]}", fontsize=9)
        for j, img in enumerate(r["t2i_top"][q][:5]):
            ax = axes[row, j + 1]
            ax.imshow(Image.open(table.image_paths[img]).convert("RGB"))
            ax.set_title(f"#{j + 1}" + (" ✓" if img == gt else ""), fontsize=9, color="green" if img == gt else "black")
        for ax in axes[row]:
            ax.axis("off")
    plt.tight_layout(); plt.show()

## 9. Export
Download `results.zip` (metrics, histories, ranks, comparisons — no checkpoints).

In [ ]:
import shutil
for f in glob.glob("runs/**/*.pt", recursive=True):
    if not KEEP_CHECKPOINTS:
        os.remove(f)
shutil.make_archive("results", "zip", "runs")
print("wrote results.zip", round(os.path.getsize("results.zip") / 1e6, 1), "MB")